# 05 - Hybrid model: Random Forest + XGBoost (weighted soft voting)

`P_hybrid = w_rf * P_RF + w_xgb * P_XGB`, default `w_rf = w_xgb = 0.5`. No claim of superiority is made beforehand; the results decide.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
import numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from src import config
config.ensure_directories()

In [2]:
from src.data.load_data import load_raw_data
from src.data.clean_data import clean_raw_dataframe, split_features_target
from src.models import model_training as mt
from src.evaluation.metrics import compute_all_metrics
df, _ = clean_raw_dataframe(load_raw_data())
X, y, ids = split_features_target(df)
X_train, X_test, y_train, y_test, ids_train, ids_test = mt.split_data(X, y, ids)
hybrid = mt.make_pipeline(mt.build_hybrid_estimator(y_train)).fit(X_train, y_train)
print(hybrid.named_steps['model'])

WeightedSoftVotingClassifier(random_forest=RandomForestClassifier(class_weight='balanced_subsample',
                                                                  max_depth=10,
                                                                  min_samples_leaf=5,
                                                                  n_estimators=300,
                                                                  n_jobs=-1,
                                                                  random_state=42),
                             xgboost=XGBClassifier(base_score=None,
                                                   booster=None, callbacks=None,
                                                   colsample_bylevel=None,
                                                   colsample_bynode=None,
                                                   colsample_bytree=0.8,
                                                   device=None,
                                                   ea

### Verify the formula on real predictions
The ensemble output must equal the weighted average of its members' probabilities.

In [3]:
model = hybrid.named_steps['model']
Xt = hybrid[:-1].transform(X_test)
p_rf, p_xgb = model.component_probabilities(Xt)
p_h = model.predict_proba(Xt)[:, 1]
assert np.allclose(p_h, 0.5 * p_rf + 0.5 * p_xgb)
print('P_hybrid == 0.5*P_RF + 0.5*P_XGB : OK')
pd.DataFrame({'P_RF': p_rf, 'P_XGB': p_xgb, 'P_hybrid': p_h, 'prediction': model.predict(Xt), 'actual': y_test.values}).head(8).round(3)

P_hybrid == 0.5*P_RF + 0.5*P_XGB : OK


,P_RF,P_XGB,P_hybrid,prediction,actual
0,0.053,0.056,0.055,0,0
1,0.877,0.944,0.911,1,0
2,0.153,0.287,0.220,0,0
3,0.548,0.512,0.530,1,0
4,0.037,0.012,0.024,0,0
5,0.696,0.792,0.744,1,0
6,0.576,0.760,0.668,1,0
7,0.219,0.385,0.302,0,0


In [4]:
print('Correlation between RF and XGB probabilities:', round(np.corrcoef(p_rf, p_xgb)[0, 1], 3))
print('Share of test customers where RF and XGB disagree on the class:', round(((p_rf >= .5) != (p_xgb >= .5)).mean(), 3))

Correlation between RF and XGB probabilities: 0.977
Share of test customers where RF and XGB disagree on the class: 0.059


### Weight sensitivity (cross-validation on the TRAINING set only)
Out-of-fold probabilities of each member are combined with different weights. The weight is chosen without touching the test set; the default 0.5 / 0.5 is kept unless there is a clear reason to change it.

In [5]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score
from src.models.baseline_models import build_random_forest, build_xgboost, compute_scale_pos_weight
cv = StratifiedKFold(config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)
oof_rf = cross_val_predict(mt.make_pipeline(build_random_forest()), X_train, y_train, cv=cv, method='predict_proba')[:, 1]
oof_xgb = cross_val_predict(mt.make_pipeline(build_xgboost(compute_scale_pos_weight(y_train))), X_train, y_train, cv=cv, method='predict_proba')[:, 1]
rows = [{'w_rf': w, 'w_xgb': round(1 - w, 1), 'CV ROC-AUC': roc_auc_score(y_train, w * oof_rf + (1 - w) * oof_xgb), 'CV PR-AUC': average_precision_score(y_train, w * oof_rf + (1 - w) * oof_xgb)} for w in np.arange(0, 1.01, 0.1)]
pd.DataFrame(rows).round(4)

,w_rf,w_xgb,CV ROC-AUC,CV PR-AUC
0,0.0,1.0,0.8396,0.6463
1,0.1,0.9,0.8409,0.6485
2,0.2,0.8,0.8420,0.6502
3,0.3,0.7,0.8430,0.6518
4,0.4,0.6,0.8438,0.6530
5,0.5,0.5,0.8445,0.6542
6,0.6,0.4,0.8450,0.6550
7,0.7,0.3,0.8453,0.6555
8,0.8,0.2,0.8455,0.6556
9,0.9,0.1,0.8455,0.6557


Differences between weightings are tiny, which is expected when both members already perform similarly. This supports keeping the simple equal-weight ensemble.

In [6]:
pd.Series(compute_all_metrics(y_test, p_h)).round(4).to_frame('Hybrid (test set)')

,Hybrid (test set)
Accuracy,0.7672
Precision,0.5434
Recall,0.7701
Sensitivity,0.7701
Specificity,0.7662
F1,0.6372
ROC-AUC,0.8437
PR-AUC,0.6529
MCC,0.4888
Cohen's Kappa,0.4732
